In [ ]:
# clean_pjm_open_after_dialog_split.csv is not public. It was the step 02 output (dialog rows split into left and right
# signer) joined with the single-signer rows from step 01, with speaker_side == '-' for single-signer videos.
# The final table with the speaker_id column added by this step is public as splits/pjm_data.csv in
# https://huggingface.co/datasets/croco-corp/pjm-data

In [ ]:
import pandas as pd
df = pd.read_csv('clean_pjm_open_after_dialog_split.csv')

In [ ]:
df

In [ ]:
filtered = df[df["speaker_side"] == "-"]

group_by = filtered.groupby(['age', 'sex'])

In [ ]:
group_by.count()

In [ ]:
%pip install deepface

In [ ]:
import cv2
import numpy as np
import pandas as pd
from deepface import DeepFace
from sklearn.cluster import AgglomerativeClustering
from tqdm import tqdm
import urllib.request
import os
import pickle

# --- Configuration ---
# Distance threshold for clustering (0.0 to 2.0 for cosine distance)
# Lower = stricter (fewer false positives, maybe more false negatives). 
# 0.3 - 0.4 is a good starting point for DeepFace/VGG-Face.
DISTANCE_THRESHOLD = 0.35 
FRAMES_TO_SAMPLE = 5  # Number of frames to average per video
EMBEDDINGS_FILE = "video_embeddings.pkl"

def get_video_embedding(video_url, num_frames=FRAMES_TO_SAMPLE):
    """
    Downloads/streams video, extracts frames, computes face embeddings, 
    and returns the average embedding vector.
    """
    try:
        cap = cv2.VideoCapture(video_url)
        if not cap.isOpened():
            return None

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0: return None
        
        # Sample frames evenly throughout the video
        frame_indices = np.linspace(0, total_frames-1, num_frames, dtype=int)
        
        embeddings = []
        
        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret: continue
            
            try:
                # DeepFace expects BGR (OpenCV default) or RGB. 
                # result is a list of dicts. We take the first face found.
                # enforce_detection=False allows it to skip frames without faces purely.
                result = DeepFace.represent(
                    frame, 
                    model_name="VGG-Face", 
                    enforce_detection=False,
                    detector_backend="opencv" # 'opencv' is fast, 'mtcnn' or 'retinaface' are more accurate but slower
                )
                
                if result:
                    # Take the largest face if multiple are found, or just the first
                    embedding = result[0]["embedding"]
                    embeddings.append(embedding)
            except:
                continue
                
        cap.release()
        
        if not embeddings:
            return None
            
        # Average the embeddings to get a stable representation for the video
        return np.mean(embeddings, axis=0)
    except Exception as e:
        print(f"Error processing {video_url}: {e}")
        return None

# Load existing embeddings if available
video_embeddings_map = {}
if os.path.exists(EMBEDDINGS_FILE):
    print(f"Loading existing embeddings from {EMBEDDINGS_FILE}...")
    with open(EMBEDDINGS_FILE, 'rb') as f:
        video_embeddings_map = pickle.load(f)

# Make a copy to avoid SettingWithCopy warnings
working_df = filtered.copy()

# Initialize a new column for the ID
working_df['generated_speaker_id'] = None

# Iterate through each (Age, Sex) group
for (age, sex), group_df in tqdm(working_df.groupby(['age', 'sex'])):
    print(f"Processing group: Age {age}, Sex {sex}, Count: {len(group_df)}")
    
    group_indices = group_df.index
    group_embeddings = []
    valid_indices = []
    
    # 1. Extract Embeddings for this group
    for i, idx in enumerate(group_indices):
        url = working_df.loc[idx, 'video_url']
        video_id = working_df.loc[idx, 'video_id']
        
        # Use cached embedding if available
        if video_id in video_embeddings_map:
            emb = video_embeddings_map[video_id]
        else:
            emb = get_video_embedding(url)
            if emb is not None:
                video_embeddings_map[video_id] = emb
        
        if emb is not None:
            group_embeddings.append(emb)
            valid_indices.append(idx)
        else:
            print(f"Could not extract face from video {video_id}")
        
        print(f"{i+1} video processed")

    if not group_embeddings:
        continue
        
    # 2. Cluster the embeddings
    if len(group_embeddings) > 1:
        X = np.array(group_embeddings)
        
        # Agglomerative Clustering is good for "grouping similar items" with a cutoff
        clustering = AgglomerativeClustering(
            n_clusters=None,
            metric='cosine', 
            linkage='average',
            distance_threshold=DISTANCE_THRESHOLD
        )
        labels = clustering.fit_predict(X)
        
        # 3. Assign IDs
        for i, idx in enumerate(valid_indices):
            cluster_id = labels[i]
            # Create a unique ID string: e.g., "F_18-30_01"
            unique_id = f"{sex}_{age}_{cluster_id:03d}"
            working_df.loc[idx, 'generated_speaker_id'] = unique_id
            
    elif len(group_embeddings) == 1:
        # If only one video with face in the group, it's a unique person
        idx = valid_indices[0]
        unique_id = f"{sex}_{age}_000"
        working_df.loc[idx, 'generated_speaker_id'] = unique_id

# Save Embeddings to Pickle
with open(EMBEDDINGS_FILE, 'wb') as f:
    pickle.dump(video_embeddings_map, f)
print(f"Saved {len(video_embeddings_map)} embeddings to {EMBEDDINGS_FILE}")

# Display results
working_df[['video_id', 'age', 'sex', 'generated_speaker_id']].head(20)

In [ ]:
import cv2
import numpy as np
import math
import json
import os

# 1. Filter for rows that actually have a generated ID
assigned_df = working_df[working_df['generated_speaker_id'].notna()]
assigned_df = assigned_df[(assigned_df['age'] == '60+') & (assigned_df['sex'] == 'M')]
grouped_speakers = assigned_df.groupby('generated_speaker_id')

print(f"Found {len(grouped_speakers)} unique speakers.")

# 2. Export to JSON
json_output = []
for speaker_id, group in grouped_speakers:
    # Ensure JSON serializable types
    video_ids = [int(x) for x in group['video_id'].values]
    json_output.append({
        "person_id": str(speaker_id),
        "video_ids": video_ids
    })

output_json_path = 'speaker_groups.json'
with open(output_json_path, 'w') as f:
    json.dump(json_output, f, indent=2)
print(f"Saved groups to {output_json_path}")

# 3. Create Big Image Canvas
TILE_SIZE = 200  # px
COLS = 10
HEADER_HEIGHT = 40
speaker_blocks = []

print("Generating visualization canvas...")
for speaker_id, group in grouped_speakers:
    n_videos = len(group)
    rows = math.ceil(n_videos / COLS)
    
    # Create canvas for this speaker
    # Height = Header + Rows * Tile
    block_height = HEADER_HEIGHT + (rows * TILE_SIZE)
    block_width = COLS * TILE_SIZE
    
    # White background for the block
    speaker_block = np.ones((block_height, block_width, 3), dtype=np.uint8) * 255
    
    # Add Header Text
    cv2.putText(speaker_block, f"Speaker: {speaker_id} (count: {n_videos})", 
                (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)
    
    # Place images
    for i, (idx, row) in enumerate(group.iterrows()):
        r = i // COLS
        c = i % COLS
        
        y_offset = HEADER_HEIGHT + (r * TILE_SIZE)
        x_offset = c * TILE_SIZE
        
        try:
            # Capture just the first frame
            cap = cv2.VideoCapture(row['video_url'])
            if cap.isOpened():
                ret, frame = cap.read()
                cap.release()
                if ret:
                    # Resize to tile size
                    frame = cv2.resize(frame, (TILE_SIZE, TILE_SIZE))
                    
                    # Add video_id overlay
                    cv2.putText(frame, str(row['video_id']), (5, TILE_SIZE-10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                    
                    # Insert frame into block
                    speaker_block[y_offset:y_offset+TILE_SIZE, x_offset:x_offset+TILE_SIZE] = frame
        except Exception:
            pass # Leave whitespace on error

    speaker_blocks.append(speaker_block)

# Stack all speaker blocks vertically
if speaker_blocks:
    full_canvas = np.vstack(speaker_blocks)
    output_img_path = 'speaker_clusters_visualization.jpg'
    cv2.imwrite(output_img_path, full_canvas)
    print(f"Saved visualization to {output_img_path} (Size: {full_canvas.shape})")
else:
    print("No groups to visualize.")

In [ ]:
# Colab only: we mounted Google Drive here to keep intermediate files between sessions. Not needed elsewhere.

In [ ]:
# Optional: restore video_embeddings.pkl (the embedding cache written by the clustering cell above) from an earlier run.
# The cache file is not public; without it the embeddings are recomputed.

In [ ]:
# Optional: save speaker_groups.json, speaker_clusters_visualization.jpg and video_embeddings.pkl for later sessions.